# CKPT7C: GRU + Static Features (6‑Month Sequences)

This notebook matches CKPT2’s **6‑month observation window** and adds **static features** (`freq`, `monetary_total`, `latetime`) to a GRU sequence model. This provides a **fair, apples‑to‑apples comparison** with CKPT2 baselines.


In [1]:
# Setup
import os, sys, random
import numpy as np
import pandas as pd

sys.path.append('..')

seed = 42
random.seed(seed)
np.random.seed(seed)


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
print('Torch:', torch.__version__)


Torch: 2.11.0+cpu


In [3]:
from pathlib import Path
from src.data import load_online_retail_ii, clean_data
from src.features import create_temporal_splits_multi_extended
from src.baselines import evaluate_model


## Data + Temporal Splits (6‑month observation)
Uses the exact same split strategy as CKPT2.


In [4]:
file_2009 = Path('../data/Year 2009-2010.csv')
file_2010 = Path('../data/Year 2010-2011.csv')

if not (file_2009.exists() and file_2010.exists()):
    raise FileNotFoundError('Raw dataset files missing in ../data/')

raw = load_online_retail_ii(file_2009, file_2010)
clean = clean_data(raw)

obs_months = 6
horizon_months = 3

train_cutoffs = ['2010-06-01','2010-09-01','2010-12-01','2011-03-01']
val_cutoff = '2011-06-01'
test_cutoff = '2011-09-01'

train_df, val_df, test_df = create_temporal_splits_multi_extended(
    clean, train_cutoffs, val_cutoff, test_cutoff,
    obs_months=obs_months, horizon_months=horizon_months
)

print('Train/Val/Test:', train_df.shape, val_df.shape, test_df.shape)


Loading Year 2009-2010...
  Loaded 525,461 rows
Loading Year 2010-2011...
  Loaded 541,910 rows

Total rows: 1,067,371

DATA CLEANING PIPELINE
Initial rows: 1,067,371
✓ Removed missing CustomerID: 243,007 rows removed
✓ Removed cancellations: 18,744 rows removed
✓ Removed invalid Quantity/Price: 71 rows removed
✓ Converted InvoiceDate to datetime
✓ Removed duplicates: 26,124 rows removed
Final rows: 779,425 (73.0% retained)
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique customers: 5,878
Unique invoices: 36,969


CREATING TEMPORAL SPLITS (MULTI-CUTOFF TRAIN, EXTENDED)

[1/3] Training Set (Multiple Cutoffs)

--- Train cutoff: 2010-06-01 ---

Creating window (extended):
  Observation: 2009-12-01 to 2010-06-01 (6 months)
  Horizon: 2010-06-01 to 2010-09-01 (3 months)
  Observation transactions: 161,616
  Horizon transactions: 83,360
  Customers in observation: 2,703
  Final customers with features: 2,703
  Customers with target > 0: 1,344 (49.7%)

--- Train cutoff: 2010-09-0

## Build Monthly Count Sequences (6 months)

In [5]:
monthly = clean.copy()
monthly['month'] = monthly['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
monthly_counts = monthly.groupby(['CustomerID', 'month']).size().unstack(fill_value=0)
monthly_counts.columns = pd.to_datetime(monthly_counts.columns)

seq_cols = [f'm{i}' for i in range(obs_months, 0, -1)]

def build_seq_features(df_split):
    seq_rows = []
    for _, row in df_split[['CustomerID', 'cutoff_date']].iterrows():
        cid = row['CustomerID']
        cutoff = pd.to_datetime(row['cutoff_date'])
        months = pd.date_range(cutoff - pd.DateOffset(months=obs_months), periods=obs_months, freq='MS')
        if cid in monthly_counts.index:
            counts = monthly_counts.loc[cid].reindex(months, fill_value=0).values
        else:
            counts = [0] * obs_months
        seq_rows.append(counts)
    return pd.DataFrame(seq_rows, columns=seq_cols, index=df_split.index)

X_train_seq = build_seq_features(train_df)
X_val_seq = build_seq_features(val_df)
X_test_seq = build_seq_features(test_df)

print('Sequence shape:', X_train_seq.shape)


Sequence shape: (12426, 6)


## Static Features (freq, monetary, recency)

In [6]:
static_cols = []
if 'freq' in train_df.columns:
    static_cols.append('freq')
if 'monetary_total' in train_df.columns:
    static_cols.append('monetary_total')
elif 'monetary_avg_invoice' in train_df.columns:
    static_cols.append('monetary_avg_invoice')
if 'latetime' in train_df.columns:
    static_cols.append('latetime')

if len(static_cols) < 2:
    raise ValueError(f'Not enough static features found. Available: {train_df.columns.tolist()}')

X_train_static = train_df[static_cols].copy()
X_val_static = val_df[static_cols].copy()
X_test_static = test_df[static_cols].copy()

y_train = train_df['target'].values
y_val = val_df['target'].values
y_test = test_df['target'].values

print('Static cols:', static_cols)


Static cols: ['freq', 'monetary_total', 'latetime']


## Dataset + DataLoader

In [7]:
from torch.utils.data import Dataset, DataLoader

stat_mean = X_train_static.mean(axis=0)
stat_std = X_train_static.std(axis=0).replace(0, 1.0)

X_train_static_n = (X_train_static - stat_mean) / stat_std
X_val_static_n = (X_val_static - stat_mean) / stat_std
X_test_static_n = (X_test_static - stat_mean) / stat_std

class SeqStaticDataset(Dataset):
    def __init__(self, X_seq, X_static, y):
        self.X_seq = torch.tensor(X_seq.values, dtype=torch.float32)
        self.X_static = torch.tensor(X_static.values, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X_seq)
    def __getitem__(self, idx):
        return self.X_seq[idx].unsqueeze(-1), self.X_static[idx], self.y[idx]

train_ds = SeqStaticDataset(X_train_seq, X_train_static_n, y_train)
val_ds = SeqStaticDataset(X_val_seq, X_val_static_n, y_val)
test_ds = SeqStaticDataset(X_test_seq, X_test_static_n, y_test)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)


## GRU + Static Head

In [8]:
class GRUStaticRegressor(nn.Module):
    def __init__(self, hidden_size=32, static_dim=3, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=hidden_size, num_layers=1, batch_first=True, dropout=0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size + static_dim, 1)
    def forward(self, x_seq, x_static):
        out, _ = self.gru(x_seq)
        last = out[:, -1, :]
        last = self.dropout(last)
        x = torch.cat([last, x_static], dim=1)
        yhat = self.fc(x)
        return F.relu(yhat).squeeze(-1)

model = GRUStaticRegressor(hidden_size=32, static_dim=len(static_cols), dropout=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.L1Loss()


## Train + Validate

In [9]:
def evaluate_loader(loader):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for xb, xs, yb in loader:
            yhat = model(xb, xs)
            preds.append(yhat.cpu().numpy())
            trues.append(yb.cpu().numpy())
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    return evaluate_model(trues, preds, 'GRU_Static')

best_val = float('inf')
best_state = None
patience = 5
no_improve = 0

epochs = 30
for epoch in range(1, epochs + 1):
    model.train()
    for xb, xs, yb in train_loader:
        optimizer.zero_grad()
        yhat = model(xb, xs)
        loss = criterion(yhat, yb)
        loss.backward()
        optimizer.step()

    val_metrics = evaluate_loader(val_loader)
    val_mae = val_metrics['MAE']
    print(f"Epoch {epoch:02d} | Val MAE: {val_mae:.4f}")

    if val_mae < best_val:
        best_val = val_mae
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print('Early stopping')
            break

if best_state is not None:
    model.load_state_dict(best_state)


Epoch 01 | Val MAE: 0.9940
Epoch 02 | Val MAE: 0.9190
Epoch 03 | Val MAE: 0.9027
Epoch 04 | Val MAE: 0.8985
Epoch 05 | Val MAE: 0.8802
Epoch 06 | Val MAE: 0.8781
Epoch 07 | Val MAE: 0.8755
Epoch 08 | Val MAE: 0.8717
Epoch 09 | Val MAE: 0.8654
Epoch 10 | Val MAE: 0.8659
Epoch 11 | Val MAE: 0.8699
Epoch 12 | Val MAE: 0.8562
Epoch 13 | Val MAE: 0.8480
Epoch 14 | Val MAE: 0.8642
Epoch 15 | Val MAE: 0.8509
Epoch 16 | Val MAE: 0.8450
Epoch 17 | Val MAE: 0.8478
Epoch 18 | Val MAE: 0.8406
Epoch 19 | Val MAE: 0.8393
Epoch 20 | Val MAE: 0.8515
Epoch 21 | Val MAE: 0.8456
Epoch 22 | Val MAE: 0.8353
Epoch 23 | Val MAE: 0.8338
Epoch 24 | Val MAE: 0.8362
Epoch 25 | Val MAE: 0.8266
Epoch 26 | Val MAE: 0.8339
Epoch 27 | Val MAE: 0.8275
Epoch 28 | Val MAE: 0.8311
Epoch 29 | Val MAE: 0.8260
Epoch 30 | Val MAE: 0.8275


## Test Evaluation + Baseline Comparison

In [10]:
test_metrics = evaluate_loader(test_loader)
print('GRU+Static Test:', test_metrics)

baseline_path = Path('../results/ckpt2/baseline_metrics.json')
if baseline_path.exists():
    import json
    baseline = json.loads(baseline_path.read_text())
    best_name, best_mae = None, float('inf')
    for name, m in baseline.items():
        if m['MAE'] < best_mae:
            best_mae = m['MAE']
            best_name = name
    print(f"Best CKPT2 baseline: {best_name} MAE={best_mae:.6f}")
    delta = test_metrics['MAE'] - best_mae
    print(f"GRU+Static vs best baseline: {delta:+.6f}")


GRU+Static Test: {'Model': 'GRU_Static', 'MAE': 1.1528733968734741, 'RMSE': np.float64(2.54774853288595)}
Best CKPT2 baseline: results_rf_test MAE=1.106082
GRU+Static vs best baseline: +0.046791
